## Imports

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
# import zipfile
# with zipfile.ZipFile('../data/raw2/Archive.zip', 'r') as zip_ref:
#     zip_ref.extractall('../data/raw2/')

## Load the dataset

In [87]:
df_orders = pd.read_csv('../data/raw/orders.csv')

In [42]:
# Reading data
df_orders_products_train = pd.read_csv('../data/raw/order_products_train.csv')

In [63]:
df_products = pd.read_csv("../data/raw/products.csv")

In [39]:
df_departments = pd.read_csv('../data/processed/departments_t.csv')

In [40]:
df_aisles = pd.read_csv('../data/raw/aisles.csv')

## Exploring the data

### orders

In [88]:
print(df_orders.columns)
print(df_orders.shape)
df_orders.head()

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='str')
(3421083, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


In [15]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 182.7 MB


In [20]:
df_orders[['order_number', 'order_dow','order_hour_of_day', 'days_since_prior_order']].describe()

,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,"3,421,083.00","3,421,083.00","3,421,083.00","3,214,874.00"
mean,17.15,2.78,13.45,11.11
std,17.73,2.05,4.23,9.21
min,1.00,0.00,0.00,0.00
25%,5.00,1.00,10.00,4.00
50%,11.00,3.00,13.00,7.00
75%,23.00,5.00,16.00,15.00
max,100.00,6.00,23.00,30.00


In [21]:
# missing values
df_orders.isna().sum()

order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

#### Analytical questions for`orders`

In [ ]:
# Exploring the eval_set distribution
df_orders['eval_set'].value_counts()

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64

In [24]:
# How many unique orders exist?
df_orders["order_id"].nunique()

3421083

In [25]:
# How many distinct customers are represented in the dataset?
df_orders["user_id"].nunique()

206209

In [26]:
# Orders by day of week
df_orders["order_dow"].value_counts().sort_index()

order_dow
0    600905
1    587478
2    467260
3    436972
4    426339
5    453368
6    448761
Name: count, dtype: int64

In [27]:
# Orders by hour of day
df_orders["order_hour_of_day"].value_counts().sort_index()

order_hour_of_day
0      22758
1      12398
2       7539
3       5474
4       5527
5       9569
6      30529
7      91868
8     178201
9     257812
10    288418
11    284728
12    272841
13    277999
14    283042
15    283639
16    272553
17    228795
18    182912
19    140569
20    104292
21     78109
22     61468
23     40043
Name: count, dtype: int64

In [28]:
# Understanding missing values in days_since_prior_order
df_orders['days_since_prior_order'].isna().sum()

np.int64(206209)

In [ ]:
# Logical check: first orders and null values
df_orders.loc[df_orders["order_number"] == 1, "days_since_prior_order"].isna().all()

np.True_

In [30]:
# Is order_id unique?
df_orders["order_id"].is_unique

True

In [31]:
# Is order_dow in the correct range?
df_orders["order_dow"].between(0, 6).all()

np.True_

In [32]:
# Is order_hour_of_day in the correct range?
df_orders["order_hour_of_day"].between(0, 23).all()

np.True_

#### Filtering the `orders` table to train orders

In [33]:
df_orders_train = df_orders[df_orders["eval_set"] == "train"]
df_orders_train.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.00
25,1492625,2,train,15,1,11,30.00
49,2196797,5,train,5,0,11,6.00
74,525192,7,train,21,2,11,6.00
78,880375,8,train,4,1,14,10.00


In [34]:
df_orders_train.drop(columns=["eval_set"], inplace=True)
df_orders_train.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,11,4,8,14.00
25,1492625,2,15,1,11,30.00
49,2196797,5,5,0,11,6.00
74,525192,7,21,2,11,6.00
78,880375,8,4,1,14,10.00


In [35]:
## Delete df_orders to free up memory
del df_orders

### order_products_train

In [11]:
print(df_orders_products_train.columns)
print(df_orders_products_train.shape)
df_orders_products_train.head()

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')
(1384617, 4)


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [44]:
df_orders_products_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 4 columns):
 #   Column             Non-Null Count    Dtype
---  ------             --------------    -----
 0   order_id           1384617 non-null  int64
 1   product_id         1384617 non-null  int64
 2   add_to_cart_order  1384617 non-null  int64
 3   reordered          1384617 non-null  int64
dtypes: int64(4)
memory usage: 42.3 MB


In [47]:
df_orders_products_train.describe()

,order_id,product_id,add_to_cart_order,reordered
count,"1,384,617.00","1,384,617.00","1,384,617.00","1,384,617.00"
mean,"1,706,297.62","25,556.24",8.76,0.60
std,"989,732.65","14,121.27",7.42,0.49
min,1.00,1.00,1.00,0.00
25%,"843,370.00","13,380.00",3.00,0.00
50%,"1,701,880.00","25,298.00",7.00,1.00
75%,"2,568,023.00","37,940.00",12.00,1.00
max,"3,421,070.00","49,688.00",80.00,1.00


In [48]:
df_orders_products_train.isna().sum()

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64

#### Analytical questions for `order_products_train`

In [49]:
# How many line items exist?
len(df_orders_products_train)

1384617

In [50]:
# How many unique orders are represented in this table?
df_orders_products_train["order_id"].nunique()

131209

In [52]:
# How many unique products are represented in this table?
df_orders_products_train["product_id"].nunique()

39123

In [ ]:
# Exploring reorder behavior
df_orders_products_train["reordered"].value_counts(dropna=False)
df_orders_products_train["reordered"].value_counts(normalize=True)  # Reorder share

reordered
1   0.60
0   0.40
Name: proportion, dtype: float64

In [57]:

# Understanding add_to_cart_order
df_orders_products_train["add_to_cart_order"].describe()

count   1,384,617.00
mean            8.76
std             7.42
min             1.00
25%             3.00
50%             7.00
75%            12.00
max            80.00
Name: add_to_cart_order, dtype: float64

In [58]:
# Which cart positions appear most often?
df_orders_products_train["add_to_cart_order"].value_counts().head(20)

add_to_cart_order
1     131209
2     124364
3     116996
4     108963
5     100745
6      91850
7      83142
8      74601
9      66618
10     59401
11     52848
12     46814
13     41431
14     36588
15     32194
16     28363
17     24841
18     21733
19     19014
20     16541
Name: count, dtype: int64

#### Understanding repeated keys in the table

In [59]:
df_orders_products_train["order_id"].is_unique

False

In [60]:
df_orders_products_train["product_id"].is_unique

False

#### Consistency checks for the `order_products_train` table

In [61]:
df_orders_products_train["reordered"].isin([0, 1]).all()

np.True_

In [62]:
df_orders_products_train[["order_id", "product_id"]].isna().sum()

order_id      0
product_id    0
dtype: int64

### products

In [12]:
print(df_products.columns)
print(df_products.shape)
df_products.head()

Index(['product_id', 'product_name', 'aisle_id', 'department_id', 'prices'], dtype='str')
(49693, 5)


,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.80
1,2,All-Seasons Salt,104,13,9.30
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50
4,5,Green Chile Anytime Sauce,5,13,4.30


In [64]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 49693 entries, 0 to 49692
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     49693 non-null  int64  
 1   product_name   49677 non-null  str    
 2   aisle_id       49693 non-null  int64  
 3   department_id  49693 non-null  int64  
 4   prices         49693 non-null  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 1.9 MB


In [66]:
df_products.describe()

,product_id,aisle_id,department_id,prices
count,"49,693.00","49,693.00","49,693.00","49,693.00"
mean,"24,844.35",67.77,11.73,9.99
std,"14,343.72",38.32,5.85,453.52
min,1.00,1.00,1.00,1.00
25%,"12,423.00",35.00,7.00,4.10
50%,"24,845.00",69.00,13.00,7.10
75%,"37,265.00",100.00,17.00,11.20
max,"49,688.00",134.00,21.00,"99,999.00"


In [67]:
df_products.isna().sum()

product_id        0
product_name     16
aisle_id          0
department_id     0
prices            0
dtype: int64

#### Analytical questions for products

In [68]:
# How many unique products exist?
df_products["product_id"].nunique()

49686

In [70]:
#  Are product IDs unique?
df_products["product_id"].is_unique

False

In [71]:
# How many unique departments are referenced in the product table?
df_products["department_id"].nunique()

21

In [72]:
# How many unique aisles are referenced in the product table?
df_products["aisle_id"].nunique()

134

In [74]:
# Do some product names repeat?
df_products["product_name"].duplicated().sum()

np.int64(20)

#### Consistency checks for the `products` table

In [75]:
df_products["product_id"].is_unique

False

In [76]:
df_products["department_id"].isna().sum()

np.int64(0)

In [77]:
df_products["aisle_id"].isna().sum()

np.int64(0)

### departments_t

In [13]:
print(df_departments.columns)
print(df_departments.shape)
df_departments.head()

Index(['department_id', 'department'], dtype='str')
(21, 2)


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [78]:
df_departments.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 468.0 bytes


In [80]:
df_departments.isna().sum()

department_id    0
department       0
dtype: int64

#### Analytical questions for departments_table

In [81]:
# How many departments exist?
df_departments["department"].nunique()

21

In [82]:
# Are department_id values unique?
df_departments["department_id"].is_unique

True

### aisles

In [14]:
print(df_aisles.columns)
print(df_aisles.shape)
df_aisles.head()

Index(['aisle_id', 'aisle'], dtype='str')
(134, 2)


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [83]:
df_aisles.info()

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.2 KB


In [84]:
df_aisles.isna().sum()

aisle_id    0
aisle       0
dtype: int64

In [85]:
df_aisles.dtypes

aisle_id    int64
aisle         str
dtype: object

#### Analytical questions for aisles

In [89]:
## How many aisles exist?
df_aisles["aisle"].nunique()

134

In [90]:
# Are aisle_id values unique?
df_aisles["aisle_id"].is_unique

True

In [91]:
df_aisles.isna().sum()

aisle_id    0
aisle       0
dtype: int64